In [0]:
import json
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# -------------------------------------------------------------------------
# 1. READ CONFIGURATION FROM AZURE STORAGE
# -------------------------------------------------------------------------
CONFIG_PATH = "abfss://config@aze2estorage.dfs.core.windows.net/config/silver_pipeline_config.json"
WORKER_NOTEBOOK_PATH = "./transform_bronze_to_silver"

config_raw = dbutils.fs.head(CONFIG_PATH, 1024 * 1024)
config = json.loads(config_raw)

# Filter active datasets
active_datasets = [item for item in config["datasets"] if item.get("enabled", True)]
print(f"📋 Loaded {len(active_datasets)} active dataset(s) for Single-Node processing.")

# -------------------------------------------------------------------------
# 2. WORKER TASK DISPATCHER
# -------------------------------------------------------------------------
def process_dataset_single_node(item):
    folder = item["folder_name"]
    dataset = item["dataset_name"]
    start_time = datetime.now()
    
    try:
        status = dbutils.notebook.run(
            path=WORKER_NOTEBOOK_PATH,
            timeout_seconds=1800,
            arguments={
                "folder_name": folder,
                "dataset_name": dataset
            }
        )
        duration = round((datetime.now() - start_time).total_seconds(), 2)
        return {"folder": folder, "dataset": dataset, "status": "SUCCESS", "duration_s": duration, "details": status}
    
    except Exception as e:
        duration = round((datetime.now() - start_time).total_seconds(), 2)
        return {"folder": folder, "dataset": dataset, "status": "FAILED", "duration_s": duration, "details": str(e)}

# -------------------------------------------------------------------------
# 3. CONCURRENT EXECUTION (Capped at 2 Workers for Single-Node Safety)
# -------------------------------------------------------------------------
MAX_SINGLE_NODE_WORKERS = 2  # Safe max for 4 cores / 14 GB shared RAM

print(f"🚀 Running pipeline on Single Node (Concurrency: {MAX_SINGLE_NODE_WORKERS})...\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_SINGLE_NODE_WORKERS) as executor:
    future_to_dataset = {executor.submit(process_dataset_single_node, item): item for item in active_datasets}
    
    for future in as_completed(future_to_dataset):
        res = future.result()
        results.append(res)
        icon = "✅" if res["status"] == "SUCCESS" else "❌"
        print(f"{icon} [{res['status']}] {res['folder']}/{res['dataset']} ({res['duration_s']}s)")

# Free up driver memory between pipeline stages
gc.collect()

# -------------------------------------------------------------------------
# 4. REPORT & PIPELINE FAILURE CHECK
# -------------------------------------------------------------------------
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

summary_schema = StructType([
    StructField("folder", StringType()),
    StructField("dataset", StringType()),
    StructField("status", StringType()),
    StructField("duration_s", DoubleType()),
    StructField("details", StringType()),
])

summary_df = spark.createDataFrame(results, schema=summary_schema)
display(summary_df)

failed_jobs = [r for r in results if r["status"] == "FAILED"]
if failed_jobs:
    raise RuntimeError(f"❌ Silver Pipeline failed for {len(failed_jobs)} dataset(s). Check summary table above.")

print("\n🎉 Silver layer pipeline completed successfully on Single Node Cluster!")